In [ ]:
# Phase 2 — Fusion Models
# Training SKIPPED (SKIP_TRAINING=True). Architecture validation runs.
import sys, torch
sys.path.insert(0, '/content/lung-nodule-fusion-xai')
print(f"CUDA: {torch.cuda.is_available()}")


In [ ]:
# Test FusionNet (intermediate fusion) architecture
from src.models.fusion_net import FusionNet
import torch

N_RADIOMIC = 20  # replace with actual selected feature count after Step 3
model = FusionNet(n_radiomic=N_RADIOMIC, backbone_name='mobilenet_v3_large', pretrained=False)
img = torch.randn(2, 3, 64, 64)
rad = torch.randn(2, N_RADIOMIC)
out = model(img, rad)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"FusionNet output: {out.shape}, params: {n_params:.1f}M")


In [ ]:
# Test early fusion XGBoost pipeline (no training data needed for API check)
from src.fusion.early_fusion import build_early_fusion_features, train_early_fusion_xgboost
import numpy as np

# Simulate small dataset
cnn_emb = np.random.randn(20, 256)
rad_feats = np.random.randn(20, N_RADIOMIC)
X = build_early_fusion_features(cnn_emb, rad_feats)
y = np.random.randint(0, 2, 20)

clf = train_early_fusion_xgboost(X, y)
print(f"Early fusion XGBoost trained. Feature dim: {X.shape[1]}")


In [ ]:
# Test late fusion (probability averaging + stacking)
from src.fusion.late_fusion import average_fusion, stacking_fusion
import numpy as np

prob_cnn = np.random.uniform(0, 1, 20)
prob_rad = np.random.uniform(0, 1, 20)
y = np.random.randint(0, 2, 20)

avg = average_fusion(prob_cnn, prob_rad, weight_cnn=0.5)
stack_probs, meta = stacking_fusion(prob_cnn, prob_rad, y, prob_cnn * 0.9, prob_rad * 0.9, y)
print(f"Average fusion output range: [{avg.min():.3f}, {avg.max():.3f}]")
print(f"Stacking meta-learner coef: {meta.coef_}")

SKIP_TRAINING = True
if not SKIP_TRAINING:
    print("Set SKIP_TRAINING=False and implement full training loop here.")
